# Module 4 — Baselines & Modern Zero-Shot Benchmark
**Type:** [Code With Me / Instructor Demo]  
**Time:** 45 minutes  
**Job:** Establish the performance floor. Run Naive, SeasonalNaive, and AutoETS via StatsForecast cross-validation. Benchmark against `amazon/chronos-t5-mini`. Score everything and build the first leaderboard.

---
## 4.1 — Setup
**[Watch Only]**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS

from config import (
    ARTIFACT_DIR,
    WORKSHOP_SUBSET_PATH,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    INTERVAL_COVERAGE,
    FM_DEMO_SERIES_N,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast, validate_score

print("Setup complete.")

---
## 4.2 — Load Panel and Build Micro Subset
**[Watch Only]**

In [ ]:
# Load validated panel from Module 3 artifact
panel = load_checkpoint("03_validated_panel")

# Build micro subset — same selection rule as Module 3
top_series = (
    panel.groupby("unique_id")["y"]
    .sum()
    .sort_values(ascending=False)
    .head(MICRO_SUBSET_N)
    .index
)
micro = panel[panel["unique_id"].isin(top_series)].copy()

print(f"Full panel : {panel['unique_id'].nunique():,} series")
print(f"Micro panel: {micro['unique_id'].nunique():,} series ({len(micro):,} rows)")

---
## 4.3 — Why Baselines Matter
**[Watch Only]**

A baseline is not a warm-up exercise. It is a **build/don't-build decision threshold**.

**Naive:** Predicts tomorrow = today. The absolute floor. If your ML model cannot beat Naive on a structured demand dataset, something is fundamentally wrong with your feature pipeline or evaluation setup.

**SeasonalNaive:** Predicts next week = same day last week. For weekly seasonal data this is a genuinely strong model. It requires zero training time, zero infrastructure, and zero maintenance. If your ML model cannot beat SeasonalNaive, you should not build the ML model.

**AutoETS:** Automatic exponential smoothing with model selection. Captures trend and seasonality without manual tuning. The gold standard for univariate statistical forecasting at scale.

**The modern standard has shifted.** A zero-shot foundation model — one trained on billions of time series and never seen your data — is now a credible fourth baseline. If your custom pipeline cannot beat `chronos-t5-mini` out of the box, that is a signal worth taking seriously.

---
## 4.4 — Configure StatsForecast
**[Code With Me — 2 lines]**

Fill in `SeasonalNaive` and `AutoETS` with `season_length` from config. Do not hardcode the number.

In [ ]:
# Configure StatsForecast with three baseline models
# Each model that accepts season_length gets it from config — no hardcoding

models = [
    Naive(),
    SeasonalNaive(__FILL_IN__),  # pass season_length from config
    AutoETS(__FILL_IN__),        # pass season_length from config
]

sf = StatsForecast(
    models=models,
    freq="D",
    n_jobs=-1,
)

print(f"StatsForecast configured with {len(models)} models:")
for m in models:
    print(f"  - {m}")

**Expected output:**
```
StatsForecast configured with 3 models:
  - Naive()
  - SeasonalNaive(season_length=7)
  - AutoETS(season_length=7)
```

---
## 4.5 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

In [ ]:
%%time
# Cross-validation on micro subset (50 series)
# Target runtime: < 45 seconds on Colab CPU

cv_micro = sf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=REFIT,
    level=[80],  # Request 80% prediction intervals from StatsForecast natively
)

print(f"CV complete. Shape: {cv_micro.shape}")
print(f"Columns: {list(cv_micro.columns)}")
print(cv_micro.head(3).to_string())

**Expected output:**
```
CV complete. Shape: (4200, 11)  # 50 series × 3 windows × 28 days = 4,200 rows
Columns: ['ds', 'cutoff', 'y', 'Naive', 'SeasonalNaive', 'AutoETS',
          'Naive-lo-80', 'Naive-hi-80', 'SeasonalNaive-lo-80', ...]
```

---
## 4.6 — Reshape to Forecast Schema
**[Code With Me — 2 lines]**

Fill in the two lines inside `reshape_statsforecast_cv`: the `model_cols` filter expression and the column selection for each model chunk.

In [ ]:
def reshape_statsforecast_cv(cv_df: pd.DataFrame, stage: str) -> pd.DataFrame:
    """
    Reshape StatsForecast wide cross-validation output to the workshop forecast schema.
    One row per (unique_id, ds, cutoff, model) observation.
    """
    cv_df = cv_df.reset_index()  # unique_id is the index after cross_validation()

    meta_cols  = ["unique_id", "ds", "cutoff", "y"]
    model_cols = [c for c in cv_df.columns
                  if c not in meta_cols and not any(x in c for x in __FILL_IN__)]  # exclude interval columns

    records = []
    for model_name in model_cols:
        lo_col = f"{model_name}-lo-80"
        hi_col = f"{model_name}-hi-80"

        chunk = cv_df[__FILL_IN__].copy()  # select meta columns + model point column
        chunk = chunk.rename(columns={model_name: "y_hat"})
        chunk["model"] = model_name
        chunk["stage"] = stage

        if lo_col in cv_df.columns and hi_col in cv_df.columns:
            chunk["lo_80"] = cv_df[lo_col].values
            chunk["hi_80"] = cv_df[hi_col].values

        records.append(chunk)

    result = pd.concat(records, ignore_index=True)
    result["y_hat"] = result["y_hat"].clip(lower=0)
    return result


baseline_micro = reshape_statsforecast_cv(cv_micro, stage="baseline")

print(f"Reshaped forecasts: {baseline_micro.shape}")
print(f"Models: {baseline_micro['model'].unique().tolist()}")
print(f"Columns: {list(baseline_micro.columns)}")

**Expected output:**
```
Reshaped forecasts: (12600, 9)  # 4200 rows × 3 models
Models: ['Naive', 'SeasonalNaive', 'AutoETS']
Columns: ['unique_id', 'ds', 'cutoff', 'y', 'y_hat', 'model', 'stage', 'lo_80', 'hi_80']
```

---
## 4.7 — Plot Baseline Forecasts
**[Watch Only]**

In [ ]:
# Plot all three baselines on a single series for one CV window
sample_uid  = top_series[0]
sample_cut  = baseline_micro["cutoff"].unique()[-1]  # most recent window

actuals = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
plot_start = sample_cut - pd.Timedelta(days=42)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(actuals[actuals.index >= plot_start].index,
        actuals[actuals.index >= plot_start].values,
        color="#333", linewidth=1.0, label="Actual", zorder=3)

colors = {"Naive": "#E53935", "SeasonalNaive": "#1E88E5", "AutoETS": "#43A047"}
for model_name, color in colors.items():
    fcast = baseline_micro[
        (baseline_micro["unique_id"] == sample_uid) &
        (baseline_micro["cutoff"]    == sample_cut) &
        (baseline_micro["model"]     == model_name)
    ].set_index("ds")
    ax.plot(fcast.index, fcast["y_hat"], color=color, linewidth=1.5,
            linestyle="--", label=model_name, zorder=4)

ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
ax.set_title(f"Baseline Forecasts — {sample_uid} (Window 3)", fontsize=11)
ax.set_ylabel("Units sold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Expected output:** A chart showing actual demand and three forecast lines. SeasonalNaive should visually track the weekly pattern more closely than Naive.

---
## 4.8 — Chronos Live Demo
**[Instructor Demo — Watch Only]**

The instructor will demonstrate `amazon/chronos-t5-mini` syntax on 3 series. Watch the output — the pattern is the same as the series-by-series loop you saw in `src/build_offline_artifacts.py`.

Chronos is a foundation model trained by Amazon on 1 billion time series observations. It requires no training on your data — you hand it a context window and it produces a forecast distribution.

**The business standard this sets:**  
If your custom ML model cannot beat a zero-shot model that has never seen your data, you should not build the ML model.

---
## 4.9 — Load Precomputed Full-Subset Baselines
**[Watch Only]**

In [ ]:
# 🔴 RED PATH (standard) — load precomputed full-subset baseline forecasts
# Covers Naive, SeasonalNaive, AutoETS, and Chronos-t5-mini
# across all 1,000 series × 3 CV windows × 28-day horizon

baseline_full = load_checkpoint("04_baseline_forecasts")

print(f"Full-subset baselines loaded:")
print(f"  Rows   : {len(baseline_full):,}")
print(f"  Models : {sorted(baseline_full['model'].unique().tolist())}")
print(f"  Series : {baseline_full['unique_id'].nunique():,}")
print(f"  Windows: {baseline_full['cutoff'].nunique()}")

**Expected output:**
```
Full-subset baselines loaded:
  Rows   : 336,000
  Models : ['AutoETS', 'Chronos-t5-mini', 'Naive', 'SeasonalNaive']
  Series : 1,000
  Windows: 3
```

---
## 4.10 — Score the Baselines
**[Code With Me — 2 lines]**

Call `score_forecasts` on the full-subset artifact. Then load the precomputed CV scores checkpoint.

In [ ]:
from src.evaluation import score_forecasts

# Score the full-subset baselines — pooled across all windows and all series
baseline_scores = score_forecasts(
    baseline_full,
    subset_name=__FILL_IN__,  # f-string: "workshop_{WORKSHOP_SUBSET_N}"
)

# Load the precomputed CV scores artifact (used in Module 8 leaderboard merge)
baseline_cv_scores = __FILL_IN__  # load_checkpoint("04_baseline_cv_scores")

print("Baseline scores (wMAPE):")
wmape_scores = baseline_scores[baseline_scores["metric"] == "wMAPE"].sort_values("score")
print(wmape_scores[["model", "metric", "score"]].to_string(index=False))

**Expected output (values will vary by subset):**
```
Baseline scores (wMAPE):
          model metric     score
        AutoETS  wMAPE  0.XXXX
  SeasonalNaive  wMAPE  0.XXXX
Chronos-t5-mini  wMAPE  0.XXXX
          Naive  wMAPE  0.XXXX
```

---
## 4.11 — Plot the Baseline Leaderboard
**[Watch Only]**

In [ ]:
wmape_plot = baseline_scores[baseline_scores["metric"] == "wMAPE"].sort_values("score")

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    wmape_plot["model"], wmape_plot["score"],
    color=["#43A047" if i == 0 else "#90CAF9" for i in range(len(wmape_plot))],
    edgecolor="white",
)
ax.set_xlabel("wMAPE (lower = better)")
ax.set_title("Baseline Leaderboard — wMAPE (Pooled, 1,000 Series × 3 Windows)", fontsize=10)
ax.invert_yaxis()
for bar, val in zip(bars, wmape_plot["score"]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

best_baseline = wmape_plot.iloc[0]
print(f"Best baseline : {best_baseline['model']} — wMAPE = {best_baseline['score']:.4f}")
print(f"This is the floor. Every model in Modules 5 and 6 must beat this.")

**Expected output:** Horizontal bar chart with four models ranked by wMAPE ascending. Green bar = best performing baseline.

---
## 4.12 — Save the Baseline Forecasts Artifact
**[Watch Only]**

In [ ]:
# Validate and save live micro-subset results
baseline_micro_validated = validate_forecast(baseline_micro, artifact_name="04_baseline_micro")

micro_path = ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet"
baseline_micro_validated.to_parquet(micro_path, index=False)
print(f"  ✓ Micro baseline saved : {micro_path.name} ({len(baseline_micro_validated):,} rows)")
print(f"  ✓ Full baseline loaded : 04_baseline_forecasts.parquet ({len(baseline_full):,} rows)")

---
## 4.13 — The Enterprise Cliff
**[Watch Only]**

We ran three classical models and a zero-shot foundation model in under 10 minutes. The performance floor is set.

What this workshop simplified:

**On statistical baselines:**  
AutoETS runs independently on each series — it is embarrassingly parallel. Scaling from 1,000 to 100,000 SKUs is a compute and orchestration problem, not a modeling problem. But StatsForecast's `n_jobs=-1` does not translate to a distributed pipeline. At Walmart scale, this is a Spark or Ray job.

**On Chronos:**  
We ran `chronos-t5-mini` — the smallest variant in the family. The production question is not "does Chronos beat AutoETS on a benchmark?" It is: "what does GPU inference cost at 100,000 SKUs × daily cadence, and does the accuracy gain justify it?" That calculation determines whether a foundation model belongs in your stack.

**On the zero-shot result:**  
If Chronos is competitive with AutoETS without seeing any of your data, that is not a reason to deploy Chronos. It is a reason to question whether your custom feature pipeline is adding real value — or just adding maintenance cost.